# 🍇 MODELADO DE VIABILIDAD AGRÍCOLA PARA EL CULTIVO DE VID
## Proyecto: SueloVid — Blvd. 2000 (Tijuana B.C.)
**Desarrollador y Científico de Datos:** Luis Armando Triche Ramírez  
**Responsable de Mediciones de Campo:** Emili Janeht Armenta  

Este notebook presenta el pipeline completo de ciencia de datos para clasificar la viabilidad de suelo y clima utilizando un **Medidor Inteligente de Suelo 6 en 1 (Sensor ST03)**. Se emplea el algoritmo de aprendizaje supervisado **K-Neighbors Classifier (KNN)** con estandarización previa de variables y visualización multivariante completa.

--- 
## 1. Fundamentos Matemáticos y Fisiológicos

### 1.1 Cálculo del Punto de Rocío ($T_{dp}$)
Para determinar la temperatura a la cual el vapor de agua en el aire se condensa, se utiliza la **Fórmula de Buck**:

$$ \alpha(T, RH) = \frac{17.27 \cdot T}{237.7 + T} + \ln\left(\frac{RH}{100}\right) $$

$$ T_{dp} = \frac{237.7 \cdot \alpha(T, RH)}{17.27 - \alpha(T, RH)} $$

### 1.2 Estandarización de Variables ($z$)
Dado que las variables tienen magnitudes y unidades muy distintas (ej. LUX en decenas de miles vs pH en escala de 1 a 14), se aplica una estandarización para evitar que las variables de mayor rango dominen el cálculo de distancias en el clasificador KNN:

$$ z = \frac{x - \mu}{\sigma} $$

### 1.3 Algoritmo K-Neighbors Classifier (KNN)
El clasificador predice la viabilidad calculando la distancia euclidiana entre el vector de consulta y los puntos del dataset de entrenamiento:

$$ d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2} $$

In [ ]:
# ======================================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ======================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Librerías importadas de forma exitosa.")

In [ ]:
# ======================================================================
# 3. CARGA DE DATASET REAL (Bitácora de Campo)
# ======================================================================
datos_medidor = {
    'Fertilidad_uS_cm': [914, 209, 233, 202, 227, 218, 133, 337, 45, 27, 233, 115, 76, 55, 290, 103, 58, 14, 293, 126, 25, 88, 223, 41, 201, 24, 26, 31, 21, 52, 221, 292, 60, 21, 22, 13, 12, 20],
    'Humedad_Suelo_pct': [60, 66, 63, 59, 51, 57, 50, 55, 51, 52, 53, 49, 53, 16, 66, 68, 53, 60, 70, 64, 51, 50, 56, 63, 54, 31, 60, 55, 46, 52, 88, 86, 56, 46, 48, 16, 10, 50],
    'pH': [7.5, 7.0, 6.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.5, 7.0, 7.0, 7.5, 7.5, 7.0, 7.0, 7.5, 7.5, 7.5, 7.0, 7.5, 7.5, 7.0, 7.5, 7.5, 7.5, 6.0, 6.5, 7.0, 7.5, 7.5, 7.5, 7.5, 7.5],
    'Temperatura_C': [20.9, 30.0, 36.2, 26.1, 21.5, 35.6, 32.1, 26.8, 23.3, 38.1, 27.5, 26.9, 25.2, 36.0, 30.6, 26.8, 22.7, 25.3, 28.5, 24.4, 21.9, 23.3, 24.9, 21.5, 18.9, 24.6, 21.6, 21.5, 27.9, 27.8, 26.8, 23.7, 18.0, 28.6, 26.6, 20.7, 15.0, 35.5],
    'Luz_Solar_LUX': [8865, 90700, 6500, 1, 1461, 36300, 1127, 1, 3700, 47800, 1223, 3, 24050, 46100, 8047, 4, 1760, 19556, 1458, 3, 9022, 15725, 9090, 2, 8677, 21500, 1090, 1, 17910, 26425, 1180, 3, 1980, 16067, 1210, 2, 1341, 47700],
    'Humedad_Amb_pct': [52, 35, 38, 54, 54, 40, 42, 55, 53, 39, 48, 54, 53, 49, 44, 54, 56, 54, 48, 53, 53, 52, 46, 56, 53, 47, 51, 51, 43, 40, 41, 51, 53, 39, 43, 53, 54, 39]
}

df = pd.DataFrame(datos_medidor)
df.info()

In [ ]:
# ======================================================================
# 4. CÁLCULO DE PUNTO DE ROCÍO E INGENIERÍA DE CARACTERÍSTICAS
# ======================================================================
def calcular_punto_rocio(temp, hum):
    a = 17.27
    b = 237.7
    alpha = ((a * temp) / (b + temp)) + np.log(hum/100.0)
    return (b * alpha) / (a - alpha)

df['Punto_Rocio_C'] = df.apply(lambda row: calcular_punto_rocio(row['Temperatura_C'], row['Humedad_Amb_pct']), axis=1)

# Evaluación bajo criterios fisiológicos idóneos para Vid
def evaluar_vid(fila):
    ph_ok = 6.0 <= fila['pH'] <= 7.5
    temp_ok = 15.0 <= fila['Temperatura_C'] <= 25.0
    humedad_amb_baja = fila['Humedad_Amb_pct'] < 50
    buen_drenaje = fila['Humedad_Suelo_pct'] < 40
    buena_luz = fila['Luz_Solar_LUX'] > 50000

    condiciones_cumplidas = sum([ph_ok, temp_ok, humedad_amb_baja, buen_drenaje, buena_luz])
    return 1 if condiciones_cumplidas >= 3 else 0

df['Viabilidad_Vid'] = df.apply(evaluar_vid, axis=1)
print("Variables calculadas y etiquetado completado.")

--- 
## 5. Análisis Exploratorio de Datos (EDA) y Gráficos

In [ ]:
# Gráfico 1: Análisis de Parcela (pH vs Humedad del Suelo con Límites)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='pH', y='Humedad_Suelo_pct',
                hue='Viabilidad_Vid', size='Luz_Solar_LUX', sizes=(20, 250),
                palette={0: '#e74c3c', 1: '#2ecc71'}, alpha=0.8)
plt.axvline(x=6.0, color='gray', linestyle='--', label='Min pH (6.0)')
plt.axvline(x=7.5, color='gray', linestyle='--', label='Max pH (7.5)')
plt.axhline(y=40, color='blue', linestyle='--', label='Max Humedad (<40%)')
plt.title('Gráfico 1: Zona de Viabilidad para la Vid (Datos Reales)', fontsize=13, fontweight='bold')
plt.xlabel('Nivel de pH (Óptimo entre 6.0 y 7.5)', fontsize=11)
plt.ylabel('Humedad del Suelo % (Óptimo < 40%)', fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Matriz de Correlación de Pearson (Heatmap)
plt.figure(figsize=(12, 8))
correlaciones = df.corr()
sns.heatmap(correlaciones, annot=True, cmap='RdYlGn', fmt=".2f", linewidths=0.5)
plt.title('Gráfico 2: Matriz de Correlación de Variables del Suelo vs Viabilidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: Análisis Multivariante (PairPlot - Grid 7x7)
sns.set_theme(style="ticks")
pair_plot = sns.pairplot(df, hue='Viabilidad_Vid', palette={0: '#e74c3c', 1: '#2ecc71'}, 
                         diag_kind='kde', plot_kws={'alpha': 0.6})
pair_plot.fig.suptitle('Gráfico 3: Análisis de Interdependencia de Variables y Viabilidad (PairPlot)', fontsize=15, fontweight='bold', y=1.02)
plt.show()

--- 
## 6. Preprocesamiento de Datos y Clasificación con KNN

In [ ]:
# Separación de variables predictoras (X) y objetivo (y)
X = df.drop('Viabilidad_Vid', axis=1)
y = df['Viabilidad_Vid']

# Train/Test Split (80% / 20% con Semilla Fija para Reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarización de características utilizando StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Configuración y entrenamiento de K-Neighbors Classifier (k=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Evaluación del Modelo
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred) * 100

print("--- RESULTADOS DE CLASIFICACIÓN KNN ---")
print(f"Precisión General en Test: {accuracy:.2f}%")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

--- 
## 7. Reporte Ejecutivo de Viabilidad Agrícola

In [ ]:
# Función para emular el reporte formal de variables individuales
def reporte_viabilidad_agricola(fertilidad, humedad_suelo, ph, temp, luz, humedad_amb):
    nombres_columnas = ['Fertilidad_uS_cm', 'Humedad_Suelo_pct', 'pH', 'Temperatura_C', 'Luz_Solar_LUX', 'Humedad_Amb_pct']
    nueva_lectura = pd.DataFrame([[fertilidad, humedad_suelo, ph, temp, luz, humedad_amb]], columns=nombres_columnas)
    
    # Calcular punto de rocío
    nueva_lectura['Punto_Rocio_C'] = calcular_punto_rocio(temp, humedad_amb)
    
    # Escalar entrada
    lectura_escalada = scaler.transform(nueva_lectura)
    
    # Predecir con KNN
    probabilidades = knn.predict_proba(lectura_escalada)[0]
    porcentaje_exito = probabilidades[1] * 100
    
    print("="*60)
    print("📊 REPORTE EJECUTIVO DE VIABILIDAD AGRÍCOLA - BLVD 2000")
    print("="*60)
    print(f"ÍNDICE DE VIABILIDAD CALCULADO: {porcentaje_exito:.1f}%")
    
    if porcentaje_exito >= 60.0:
        print("\n✅ VEREDICTO: TERRENO ALTAMENTE RENTABLE PARA LA VID.")
        print("Recomendación: Proceder con la siembra. Condiciones de acidez y drenaje idóneas.")
    else:
        print("\n❌ VEREDICTO: TERRENO DE ALTO RIESGO / NO APTO ACTUALMENTE.")
        print("Recomendación: Detener plantío. Corregir drenaje y balance mineral mediante fertirriego.")
        
    print("\n--- DESGLOSE DE VARIABLES DEL SENSOR ST03 ---")
    print(f"1. pH del Suelo: {ph:.2f}")
    print("   -> Excelente. Permite la correcta absorción de nutrientes." if 6.0 <= ph <= 7.5 else "   -> Riesgo. Fuera del rango óptimo (6.0 - 7.5).")
    
    print(f"2. Humedad del Suelo: {humedad_suelo:.1f}%")
    print("   -> Excelente. Buen drenaje detectado, previene pudrición." if humedad_suelo < 40 else "   -> Riesgo. Suelo saturado/encharcado.")
    
    print(f"3. Temperatura Radicular: {temp:.1f}°C")
    print("   -> Excelente. Temperatura idónea para subsuelo." if 15.0 <= temp <= 25.0 else "   -> Riesgo. Fuera de confort térmico radicular.")
    
    print(f"4. Luz Solar Receptada: {luz:,} LUX")
    print("   -> Excelente. Alta radiación para fotosíntesis." if luz > 50000 else "   -> Riesgo. Zona sombreada; afectará maduración de uva.")
    
    print(f"5. Humedad Ambiental: {humedad_amb:.1f}%")
    print("   -> Excelente. Ambiente seco que evita hongos." if humedad_amb < 50 else "   -> Riesgo. Propensión a desarrollo de patógenos.")
    
    print(f"6. Fertilidad (Conductividad): {fertilidad} uS/cm")
    print("   -> Indicador base de sales minerales. Ajustar vía fertirriego.")
    print("="*60)

# Prueba con el promedio histórico real de campo
reporte_viabilidad_agricola(fertilidad=140, humedad_suelo=53.5, ph=7.3, temp=26.1, luz=12568, humedad_amb=48.4)